In [0]:
from datetime import datetime, date as _date
from pyspark.sql import functions as F, Window

spark.sql("USE workspace.steam")
MIN_DATE = "2026-10-09"

dbutils.widgets.text("start_date", MIN_DATE)
dbutils.widgets.text("end_date", "")

latest = str(spark.sql("SELECT MAX(load_date) FROM bronze_steam").first()[0])
start_date = dbutils.widgets.get("start_date").strip() or MIN_DATE
end_date = dbutils.widgets.get("end_date").strip() or latest

for d in (start_date, end_date):
    datetime.strptime(d, "%Y-%m-%d")  # fails on a badly written date
if start_date < MIN_DATE:
    raise ValueError(f"start_date must be on or after {MIN_DATE}")
if start_date > end_date:
    raise ValueError("start_date must be on or before end_date")
if end_date > latest:
    raise ValueError(f"end_date is after the latest loaded date ({latest})")

print(f"comparing {start_date} to {end_date} (latest loaded date: {latest})")

In [0]:
TRACKED = ["name", "developer", "publisher", "positive", "negative", "ccu",
           "owners", "price", "initialprice", "discount", "score_rank", "userscore"]

def snapshot(cutoff, inclusive=True):
    """Each game's latest Bronze row on or before (or strictly before) the cutoff date."""
    cond = (F.col("load_date") <= cutoff) if inclusive else (F.col("load_date") < cutoff)
    w = Window.partitionBy("appid").orderBy(F.col("load_date").desc())
    return (spark.table("bronze_steam")
            .where((F.col("load_date") >= MIN_DATE) & cond)
            .withColumn("rn", F.row_number().over(w)).where("rn = 1")
            .select("appid", "load_date", "row_hash", *TRACKED))

same_day = start_date == end_date
# same date: compare that day against the state just before it
before = snapshot(F.to_date(F.lit(start_date)), inclusive=not same_day)
after = snapshot(F.to_date(F.lit(end_date)))

b = before.select("appid", *[F.col(c).alias(f"b_{c}") for c in ["row_hash"] + TRACKED])
a = after.select("appid", *[F.col(c).alias(f"a_{c}") for c in ["row_hash"] + TRACKED])
cmp = a.join(b, "appid", "left").withColumn(
    "status",
    F.when(F.col("b_row_hash").isNull(), "new")
     .when(F.col("a_row_hash") != F.col("b_row_hash"), "changed")
     .otherwise("unchanged"))

baseline = f"before {start_date}" if same_day else start_date
print(f"baseline: {baseline} | games: {before.count()}")
print(f"end:      {end_date} | games: {after.count()}")
display(cmp.groupBy("status").count().orderBy("status"))

In [0]:
changed = cmp.where("status = 'changed'")

# how many changed games had each field change
field_counts = changed.select(*[
    F.sum(F.when(~F.col(f"a_{c}").eqNullSafe(F.col(f"b_{c}")), 1).otherwise(0)).alias(c)
    for c in TRACKED])
display(field_counts)

# totals across all games on each date
def totals(df, p):
    return df.select(
        F.sum(F.expr(f"try_cast({p}positive as bigint)")).alias("pos"),
        F.sum(F.expr(f"try_cast({p}negative as bigint)")).alias("neg"),
        F.sum(F.expr(f"try_cast({p}ccu as bigint)")).alias("ccu")).first()

t0 = totals(cmp.where("status <> 'new'"), "b_")
t1 = totals(cmp, "a_")
print(f"reviews:          {t0.pos + t0.neg:,} -> {t1.pos + t1.neg:,}  "
      f"(+{(t1.pos + t1.neg) - (t0.pos + t0.neg):,})")
print(f"positive reviews: +{t1.pos - t0.pos:,} | negative reviews: +{t1.neg - t0.neg:,}")
print(f"total ccu:        {t0.ccu:,} -> {t1.ccu:,}  ({t1.ccu - t0.ccu:+,})")

In [0]:
display(spark.sql("""
    SELECT load_date, name, positive, negative, ccu, price, discount
    FROM bronze_steam
    WHERE appid IN ('730', '570', '578080')
    ORDER BY appid, load_date
"""))

In [0]:
p = changed.select(
    "appid", F.col("a_name").alias("name"),
    F.expr("try_cast(b_price as double) / 100").alias("old_price"),
    F.expr("try_cast(a_price as double) / 100").alias("new_price"),
    F.expr("try_cast(b_discount as int)").alias("old_discount"),
    F.expr("try_cast(a_discount as int)").alias("new_discount"),
    F.expr("try_cast(a_ccu as int)").alias("ccu"))

display(p.select(
    F.sum(F.when(F.col("new_price") < F.col("old_price"), 1).otherwise(0)).alias("price_drops"),
    F.sum(F.when(F.col("new_price") > F.col("old_price"), 1).otherwise(0)).alias("price_rises"),
    F.sum(F.when((F.col("old_discount") == 0) & (F.col("new_discount") > 0), 1).otherwise(0)).alias("went_on_sale"),
    F.sum(F.when((F.col("old_discount") > 0) & (F.col("new_discount") == 0), 1).otherwise(0)).alias("sale_ended"),
    F.sum(F.when((F.col("old_discount") > 0) & (F.col("new_discount") > 0) &
                 (F.col("old_discount") != F.col("new_discount")), 1).otherwise(0)).alias("discount_changed")))

print("Biggest discounts that started in this period (by player count):")
display(p.where((F.col("old_discount") == 0) & (F.col("new_discount") > 0))
         .orderBy(F.col("ccu").desc()).limit(10))

In [0]:
print("New games in this period (by player count):")
display(cmp.where("status = 'new'").select(
    "appid", F.col("a_name").alias("name"), F.col("a_developer").alias("developer"),
    F.expr("try_cast(a_ccu as int)").alias("ccu"),
    F.expr("try_cast(a_positive as int) + try_cast(a_negative as int)").alias("reviews"),
    F.expr("try_cast(a_price as double) / 100").alias("price_usd"),
    F.col("a_owners").alias("owners"))
  .orderBy(F.col("ccu").desc(), F.col("reviews").desc()).limit(20))

lo = start_date if same_day else None
print("Day by day:")
first_seen = spark.table("bronze_steam").groupBy("appid").agg(F.min("load_date").alias("first_date"))
daily = (spark.table("bronze_steam")
         .where((F.col("load_date") >= (start_date if same_day else F.date_add(F.to_date(F.lit(start_date)), 1)))
                & (F.col("load_date") <= end_date))
         .join(first_seen, "appid")
         .groupBy("load_date")
         .agg(F.sum(F.when(F.col("load_date") == F.col("first_date"), 1).otherwise(0)).alias("new_games"),
              F.sum(F.when(F.col("load_date") > F.col("first_date"), 1).otherwise(0)).alias("changed_games"))
         .orderBy("load_date"))
display(daily)